# Databricks Auto Loader - Complete Tutorial

## What is Auto Loader?

Auto Loader is an **incremental data ingestion framework** in Databricks that efficiently processes new data files as they arrive in cloud storage. It uses Structured Streaming to provide exactly-once semantics and automatically infers, evolves, and tracks schemas.

### Key Features:
- **Incremental Processing**: Only processes new files, not the entire directory
- **Automatic Schema Inference & Evolution**: Detects schema changes automatically
- **Scalability**: Handles millions of files efficiently
- **Exactly-Once Semantics**: Ensures data is processed once and only once
- **Cloud-Optimized**: Works seamlessly with S3, ADLS, GCS
- **Cost-Effective**: Uses file notifications or directory listing based on your needs

### When Was It Added?
Auto Loader was introduced in **Databricks Runtime 7.3 (September 2020)** and has been continuously enhanced with new features in subsequent releases.

## Common Use Cases

### 1. **Real-Time Data Ingestion**
   - Stream logs from applications
   - Process IoT sensor data
   - Ingest clickstream events

### 2. **Data Lake Ingestion**
   - Build Bronze/Silver/Gold architectures
   - Incremental ETL pipelines
   - Data warehouse loading

### 3. **File Format Processing**
   - JSON, CSV, Parquet, Avro, ORC
   - Text files, binary files
   - Semi-structured data

### 4. **Schema Evolution Scenarios**
   - Adding new columns over time
   - Changing data types
   - Handling schema drift

### 5. **Large-Scale File Processing**
   - Processing millions of small files
   - Handling nested directory structures
   - Multi-tenant data ingestion

## Level 1: Basic Auto Loader Example

The simplest way to use Auto Loader with minimal configuration.

In [0]:
# Basic Auto Loader example for JSON files
# UPDATE THE PATH BELOW with your actual file location

file_path = "s3://nitya-cloutech/parquet/sales_suppliers"  # TODO: Update this path
checkpoint_path = "s3://nitya-cloutech/Account/schema1"

# Read streaming data using Auto Loader
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  .option("cloudFiles.schemaLocation", "s3://nitya-cloutech/Account/schema_location")
  .load(file_path)
)

# Display schema
print("Schema inferred by Auto Loader:")
df.printSchema()

# Write to Delta table
(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table("dev_account.hr.autoloader_basic_table")
)

## Level 2: Auto Loader with Schema Hints

Provide schema hints to guide Auto Loader's schema inference.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

file_path = "/path/to/your/json/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_schema_hints"

# Define schema hints
schema_hints = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaHints", "id INT, name STRING, timestamp TIMESTAMP")  # Alternative: DDL string
  # .schema(schema_hints)  # Or provide full schema
  .load(file_path)
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .table("autoloader_schema_hints_table")
)

## Level 3: Schema Evolution

Handle schema changes automatically as new files arrive with different schemas.

In [0]:
file_path = "/path/to/your/json/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_schema_evolution"
schema_location = "/tmp/autoloader_schema_location"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  
  # Schema evolution options
  .option("cloudFiles.schemaLocation", schema_location)  # Store inferred schema
  .option("cloudFiles.inferColumnTypes", "true")  # Infer types (default: false, infers as string)
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Options: addNewColumns, rescue, none
  
  # Rescue data that doesn't match schema
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # Column name for rescued data
  
  .load(file_path)
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")  # Enable schema merging in Delta
  .table("autoloader_schema_evolution_table")
)

## Level 4: File Notification Modes

Auto Loader supports two modes for detecting new files:

### 1. **File Notification Mode** (Default for most cases)
- Uses cloud provider's notification service (S3 Event Notifications, Azure Event Grid, GCS Pub/Sub)
- More scalable and cost-effective for large directories
- Slight setup complexity

### 2. **Directory Listing Mode**
- Periodically lists the directory to find new files
- Simpler setup, no cloud configuration needed
- Better for smaller directories or when notifications aren't available

In [0]:
file_path = "/path/to/your/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_notification"

# Option 1: File Notification Mode (Default)
df_notification = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.useNotifications", "true")  # Default: auto-detected
  .option("cloudFiles.queueUrl", "<sqs-queue-url>")  # For AWS, optional if auto-configured
  .option("cloudFiles.connectionString", "<connection-string>")  # For Azure Event Grid
  .load(file_path)
)

# Option 2: Directory Listing Mode
df_directory_listing = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.useNotifications", "false")  # Force directory listing
  .option("cloudFiles.maxFilesPerTrigger", "1000")  # Limit files per batch
  .load(file_path)
)

print("File notification mode configured successfully")

## Complete Auto Loader Properties Reference

### Core Properties

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.format` | Source file format | **Required** | json, csv, parquet, avro, orc, text, binaryFile |
| `cloudFiles.schemaLocation` | Path to store inferred schema | None | Any valid path |
| `cloudFiles.useNotifications` | Use file notifications vs directory listing | auto | true, false, auto |

### Schema Management

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.inferColumnTypes` | Infer data types vs treat as strings | false | true, false |
| `cloudFiles.schemaHints` | DDL string for schema hints | None | "col1 INT, col2 STRING" |
| `cloudFiles.schemaEvolutionMode` | How to handle schema changes | addNewColumns | addNewColumns, rescue, none |
| `cloudFiles.rescuedDataColumn` | Column name for rescued data | _rescued_data | Any valid column name |

### Performance Tuning

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.maxFilesPerTrigger` | Max files to process per batch | 1000 | Integer |
| `cloudFiles.maxBytesPerTrigger` | Max bytes to process per batch | None | String (e.g., "10g") |
| `cloudFiles.minBytesPerFile` | Min file size to process | None | Long (bytes) |
| `cloudFiles.maxFilesPerBatch` | (Deprecated) Use maxFilesPerTrigger | 1000 | Integer |

### File Filtering

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.pathGlobFilter` | Glob pattern to include files | None | "*.json", "data/*.csv" |
| `cloudFiles.modifiedAfter` | Only process files modified after timestamp | None | ISO-8601 timestamp |
| `cloudFiles.modifiedBefore` | Only process files modified before timestamp | None | ISO-8601 timestamp |

### Metadata Columns

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.includeExistingFiles` | Process existing files on start | true | true, false |
| `cloudFiles.validateOptions` | Validate configuration options | true | true, false |
| `cloudFiles.allowOverwrites` | Allow file overwrites | false | true, false |

### Cloud-Specific Options

| Property | Description | Default | Values |
|----------|-------------|---------|--------|
| `cloudFiles.queueUrl` | AWS SQS queue URL | auto-configured | SQS URL |
| `cloudFiles.connectionString` | Azure Event Grid connection | auto-configured | Connection string |
| `cloudFiles.region` | AWS region | auto-detected | us-west-2, etc. |

### Format-Specific Options (CSV)

| Property | Description | Default |
|----------|-------------|---------|  
| `header` | CSV has header row | false |
| `delimiter` | Field delimiter | , |
| `quote` | Quote character | " |
| `escape` | Escape character | \ |
| `inferSchema` | Infer CSV schema | false |
| `multiLine` | Support multi-line fields | false |

## Level 5: Working with Different File Formats

Auto Loader supports multiple file formats with format-specific options.

In [0]:
# CSV Files with Auto Loader
file_path = "/path/to/your/csv/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_csv"

df_csv = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  
  # CSV-specific options
  .option("header", "true")
  .option("delimiter", ",")
  .option("inferSchema", "true")
  .option("escape", "\\")
  .option("quote", '"')
  .option("multiLine", "true")
  
  # Schema evolution
  .option("cloudFiles.schemaLocation", "/tmp/csv_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  
  .load(file_path)
)

print("CSV Auto Loader configured")
df_csv.printSchema()

In [0]:
# Parquet Files
df_parquet = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  .option("cloudFiles.schemaLocation", "/tmp/parquet_schema")
  .load("/path/to/parquet/files")  # TODO: Update this path
)

# Avro Files
df_avro = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "avro")
  .option("cloudFiles.schemaLocation", "/tmp/avro_schema")
  .load("/path/to/avro/files")  # TODO: Update this path
)

# ORC Files
df_orc = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "orc")
  .option("cloudFiles.schemaLocation", "/tmp/orc_schema")
  .load("/path/to/orc/files")  # TODO: Update this path
)

# Text Files
df_text = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "text")
  .load("/path/to/text/files")  # TODO: Update this path
)

# Binary Files
df_binary = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "binaryFile")
  .load("/path/to/binary/files")  # TODO: Update this path
)

print("Multiple format Auto Loaders configured")

## Level 6: Advanced Usage with Transformations

Add metadata columns, apply transformations, and use file filtering.

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name, to_date

file_path = "/path/to/your/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_advanced"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/tmp/advanced_schema")
  
  # File filtering
  .option("cloudFiles.pathGlobFilter", "*.json")  # Only JSON files
  .option("cloudFiles.modifiedAfter", "2024-01-01T00:00:00")  # Files after Jan 1, 2024
  
  # Performance tuning
  .option("cloudFiles.maxFilesPerTrigger", "100")
  .option("cloudFiles.maxBytesPerTrigger", "1g")
  
  # Schema options
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  
  .load(file_path)
)

# Add metadata and transformations
df_transformed = (df
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_file", input_file_name())
  .withColumn("processing_date", to_date(current_timestamp()))
  .filter(col("id").isNotNull())  # Filter out null IDs
)

# Write with partitioning
(df_transformed.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .partitionBy("processing_date")  # Partition by date
  .table("autoloader_advanced_table")
)

## Level 7: Production Best Practices

Advanced patterns for production deployments.

In [0]:
from pyspark.sql.functions import col, current_timestamp, sha2, concat_ws

file_path = "/path/to/your/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_production"
schema_location = "/tmp/production_schema"

# Production-grade Auto Loader configuration
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  
  # Schema management
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "rescue")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  
  # Performance optimization
  .option("cloudFiles.maxFilesPerTrigger", "1000")
  .option("cloudFiles.maxBytesPerTrigger", "10g")
  
  # File notification (production)
  .option("cloudFiles.useNotifications", "true")
  
  # Quality controls
  .option("cloudFiles.validateOptions", "true")
  .option("cloudFiles.includeExistingFiles", "true")
  
  # File filtering
  .option("cloudFiles.pathGlobFilter", "*.json")
  
  .load(file_path)
)

# Add data quality checks and metadata
df_production = (df
  .withColumn("ingestion_time", current_timestamp())
  .withColumn("source_file", input_file_name())
  .withColumn("record_hash", sha2(concat_ws("|", col("*")), 256))  # For deduplication
  .withColumn("is_rescued", col("_rescued_data").isNotNull())
)

# Write with quality checks
(df_production.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  
  # Delta optimizations
  .option("optimizeWrite", "true")
  .option("autoCompact", "true")
  
  # Output mode
  .outputMode("append")
  
  # Trigger configuration
  .trigger(processingTime="1 minute")  # Process every minute
  
  # Error handling
  .option("queryName", "production_autoloader_stream")
  
  .table("autoloader_production_table")
)

## Monitoring and Troubleshooting

Track Auto Loader performance and diagnose issues.

In [0]:
# Get active streaming queries
active_streams = spark.streams.active

print(f"Number of active streams: {len(active_streams)}")

for stream in active_streams:
    print(f"\nStream ID: {stream.id}")
    print(f"Name: {stream.name}")
    print(f"Status: {stream.status}")
    print(f"Recent Progress:")
    
    # Get last progress
    if stream.lastProgress:
        progress = stream.lastProgress
        print(f"  Batch ID: {progress.get('batchId', 'N/A')}")
        print(f"  Input Rows: {progress.get('numInputRows', 'N/A')}")
        print(f"  Processing Rate: {progress.get('processedRowsPerSecond', 'N/A')} rows/sec")
        print(f"  Batch Duration: {progress.get('durationMs', {}).get('triggerExecution', 'N/A')} ms")

# Query specific stream status
# stream.status  # Current state
# stream.recentProgress  # Last few batches
# stream.lastProgress  # Most recent batch
# stream.exception  # If any error occurred

In [0]:
from pyspark.sql.functions import explode, col

file_path = "/path/to/nested/json/files"  # TODO: Update this path

# Handle nested JSON with Auto Loader
df_nested = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/tmp/nested_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  
  # For multi-line JSON
  .option("multiLine", "true")
  
  .load(file_path)
)

# Flatten nested structures
df_flattened = (df_nested
  .withColumn("ingestion_time", current_timestamp())
  # Example: explode nested array
  # .withColumn("item", explode(col("items")))
  # .select("id", "item.*", "ingestion_time")
)

print("Nested JSON Auto Loader configured")
df_flattened.printSchema()

In [0]:
from pyspark.sql.functions import col, window

file_path = "/path/to/your/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_watermark"

# Auto Loader with watermarking for handling late-arriving data
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/tmp/watermark_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  .load(file_path)
)

# Add watermark for late data handling (e.g., 1 hour tolerance)
df_with_watermark = (df
  .withWatermark("event_timestamp", "1 hour")  # Assumes 'event_timestamp' column exists
  .groupBy(
    window(col("event_timestamp"), "10 minutes"),
    col("user_id")
  )
  .count()
)

# Write aggregated results
(df_with_watermark.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .outputMode("update")  # Use update mode for aggregations
  .table("autoloader_watermark_table")
)

## Common Issues and Troubleshooting

### Issue 1: Schema Inference Problems
**Problem**: Schema is not inferred correctly or columns have wrong types

**Solutions**:
- Use `cloudFiles.inferColumnTypes = true` to infer proper types
- Provide schema hints: `.option("cloudFiles.schemaHints", "id INT, price DOUBLE")`
- Use explicit schema: `.schema(your_schema)`

### Issue 2: Files Not Being Picked Up
**Problem**: New files are added but not processed

**Solutions**:
- Check `cloudFiles.includeExistingFiles` setting
- Verify file notification configuration (SQS queue, Event Grid)
- Use directory listing mode: `.option("cloudFiles.useNotifications", "false")`
- Check path glob filter: `.option("cloudFiles.pathGlobFilter", "*.json")`

### Issue 3: Schema Evolution Errors
**Problem**: Stream fails when schema changes

**Solutions**:
- Set schema evolution mode: `.option("cloudFiles.schemaEvolutionMode", "rescue")`
- Enable merge schema in Delta: `.option("mergeSchema", "true")`
- Use rescued data column to capture incompatible records

### Issue 4: Performance Issues
**Problem**: Stream is slow or processes too many files at once

**Solutions**:
- Limit files per trigger: `.option("cloudFiles.maxFilesPerTrigger", "100")`
- Limit bytes per trigger: `.option("cloudFiles.maxBytesPerTrigger", "1g")`
- Use file notifications instead of directory listing
- Optimize trigger interval: `.trigger(processingTime="5 minutes")`

### Issue 5: Checkpoint Issues
**Problem**: Stream fails to restart or checkpoint is corrupted

**Solutions**:
- Use a new checkpoint location
- Ensure checkpoint path is accessible and has proper permissions
- Don't share checkpoint locations between different streams

### Issue 6: Permission Errors
**Problem**: Access denied errors when reading files

**Solutions**:
- Verify IAM roles/permissions for S3, ADLS, or GCS
- Check if cluster has proper instance profile or service principal
- Verify SQS queue permissions (for AWS)
- Check Event Grid subscription permissions (for Azure)

## Best Practices and Tips

### 1. **Always Use Checkpoint Locations**
- Never run Auto Loader without a checkpoint location
- Use unique checkpoint paths for each stream
- Store checkpoints in a reliable, persistent location

### 2. **Schema Management**
- Always specify `cloudFiles.schemaLocation` for production
- Use `inferColumnTypes = true` for proper type inference
- Consider using schema hints for critical columns
- Plan for schema evolution from day one

### 3. **Performance Optimization**
- Start with file notifications (`useNotifications = true`)
- Tune `maxFilesPerTrigger` based on your file sizes
- Use `maxBytesPerTrigger` to limit memory usage
- Monitor stream metrics regularly

### 4. **Error Handling**
- Use `schemaEvolutionMode = rescue` to capture problematic records
- Add a `_rescued_data` column for troubleshooting
- Implement data quality checks in your transformations
- Monitor for rescued records

### 5. **Cost Optimization**
- Use directory listing for small workloads (< 10K files)
- Use file notifications for large-scale ingestion
- Set appropriate trigger intervals
- Clean up old checkpoint files periodically

### 6. **Testing**
- Test with `includeExistingFiles = false` initially
- Validate schema inference before production
- Test schema evolution scenarios
- Monitor first few batches closely

### 7. **Monitoring**
- Use `spark.streams.active` to check stream health
- Monitor `lastProgress` for performance metrics
- Set up alerts for stream failures
- Track processing latency and throughput

### 8. **Security**
- Use proper IAM roles/service principals
- Encrypt checkpoint locations
- Limit access to source and target locations
- Audit access patterns

In [0]:
# Managing Auto Loader Streams

# 1. List all active streams
print("Active Streams:")
for stream in spark.streams.active:
    print(f"  - ID: {stream.id}, Name: {stream.name}")

# 2. Stop a specific stream by name
def stop_stream_by_name(stream_name):
    for stream in spark.streams.active:
        if stream.name == stream_name:
            print(f"Stopping stream: {stream_name}")
            stream.stop()
            return True
    print(f"Stream {stream_name} not found")
    return False

# Example: stop_stream_by_name("production_autoloader_stream")

# 3. Stop all active streams
def stop_all_streams():
    print(f"Stopping {len(spark.streams.active)} active streams...")
    for stream in spark.streams.active:
        print(f"  Stopping: {stream.name}")
        stream.stop()
    print("All streams stopped")

# Example: stop_all_streams()

# 4. Get detailed status of a stream
def get_stream_status(stream_name):
    for stream in spark.streams.active:
        if stream.name == stream_name:
            print(f"Stream: {stream.name}")
            print(f"Status: {stream.status}")
            print(f"Is Active: {stream.isActive}")
            if stream.lastProgress:
                print(f"Last Batch ID: {stream.lastProgress.get('batchId')}")
                print(f"Input Rows: {stream.lastProgress.get('numInputRows')}")
            return stream.status
    print(f"Stream {stream_name} not found")
    return None

# Example: get_stream_status("production_autoloader_stream")

# 5. Wait for stream termination (useful in notebooks)
# stream.awaitTermination()  # Wait indefinitely
# stream.awaitTermination(timeout=60)  # Wait for 60 seconds

print("Stream management functions loaded")

## Advanced: Auto Loader with Delta Lake Features

Combine Auto Loader with Delta Lake's powerful features for optimal data lakehouse architecture.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp

file_path = "/path/to/your/files"  # TODO: Update this path
checkpoint_path = "/tmp/autoloader_checkpoint_merge"
target_table = "autoloader_merge_table"

# Read with Auto Loader
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/tmp/merge_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  .load(file_path)
)

# Add processing metadata
df_processed = df.withColumn("updated_at", current_timestamp())

# Function to perform merge (upsert) operation
def merge_to_delta(microBatchDF, batchId):
    # Create or get Delta table
    if DeltaTable.isDeltaTable(spark, f"spark_catalog.default.{target_table}"):
        delta_table = DeltaTable.forName(spark, f"default.{target_table}")
        
        # Perform merge (upsert)
        delta_table.alias("target").merge(
            microBatchDF.alias("source"),
            "target.id = source.id"  # Match condition
        ).whenMatchedUpdate(
            set = {
                "name": "source.name",
                "value": "source.value",
                "updated_at": "source.updated_at"
            }
        ).whenNotMatchedInsert(
            values = {
                "id": "source.id",
                "name": "source.name",
                "value": "source.value",
                "updated_at": "source.updated_at"
            }
        ).execute()
    else:
        # First batch - create table
        microBatchDF.write.format("delta").mode("append").saveAsTable(target_table)

# Write using foreachBatch for merge operations
(df_processed.writeStream
  .foreachBatch(merge_to_delta)
  .option("checkpointLocation", checkpoint_path)
  .trigger(processingTime="1 minute")
  .start()
)

## Quick Reference Guide

### Basic Syntax
```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "<format>")
  .load("<path>")
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", "<checkpoint>")
  .table("<table_name>")
)
```

### Essential Options

| Option | Purpose | Example |
|--------|---------|--------|
| `cloudFiles.format` | File format | json, csv, parquet |
| `cloudFiles.schemaLocation` | Schema storage | "/path/to/schema" |
| `cloudFiles.inferColumnTypes` | Type inference | "true" |
| `cloudFiles.schemaEvolutionMode` | Schema changes | addNewColumns, rescue |
| `cloudFiles.maxFilesPerTrigger` | Rate limiting | "100" |
| `cloudFiles.pathGlobFilter` | File filtering | "*.json" |

### File Formats
- **JSON**: `.option("cloudFiles.format", "json")`
- **CSV**: `.option("cloudFiles.format", "csv").option("header", "true")`
- **Parquet**: `.option("cloudFiles.format", "parquet")`
- **Avro**: `.option("cloudFiles.format", "avro")`
- **Text**: `.option("cloudFiles.format", "text")`

### Common Patterns

**1. Schema Inference**
```python
.option("cloudFiles.schemaLocation", "/path")
.option("cloudFiles.inferColumnTypes", "true")
```

**2. Schema Evolution**
```python
.option("cloudFiles.schemaEvolutionMode", "rescue")
.option("cloudFiles.rescuedDataColumn", "_rescued_data")
```

**3. Performance Tuning**
```python
.option("cloudFiles.maxFilesPerTrigger", "100")
.option("cloudFiles.maxBytesPerTrigger", "1g")
```

**4. File Filtering**
```python
.option("cloudFiles.pathGlobFilter", "*.json")
.option("cloudFiles.modifiedAfter", "2024-01-01T00:00:00")
```

### Monitoring Commands
```python
# List active streams
spark.streams.active

# Stop a stream
stream.stop()

# Get stream status
stream.status
stream.lastProgress
```

### Remember:
1. ✅ Always use checkpoint locations
2. ✅ Store schema in schemaLocation for production
3. ✅ Enable inferColumnTypes for proper types
4. ✅ Plan for schema evolution
5. ✅ Monitor stream health regularly
6. ✅ Test with small datasets first
7. ✅ Use file notifications for large-scale ingestion

## Additional Resources

### Official Documentation
- [Databricks Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- [Delta Lake Documentation](https://docs.delta.io/latest/index.html)

### Key Features by Databricks Runtime Version

| DBR Version | Key Auto Loader Features |
|-------------|-------------------------|
| 7.3+ | Auto Loader introduced, basic file notification support |
| 8.2+ | Schema evolution improvements, rescued data column |
| 9.1+ | Enhanced performance, better cloud integration |
| 10.4+ | Advanced schema hints, improved error handling |
| 11.3+ | Optimized directory listing, better monitoring |
| 12.0+ | Enhanced file notification, Unity Catalog integration |
| 13.0+ | Performance improvements, additional file formats |

### When to Use Auto Loader vs Alternatives

**Use Auto Loader when:**
- Processing files incrementally as they arrive
- Handling millions of files
- Need automatic schema evolution
- Want exactly-once processing guarantees
- Building streaming ETL pipelines

**Consider alternatives when:**
- One-time bulk file processing → Use `spark.read` with wildcards
- Small number of files → Regular batch processing may suffice
- Need complex file processing logic → Consider custom streaming source

### Performance Tips Summary
1. Use file notifications for directories with > 10,000 files
2. Set appropriate `maxFilesPerTrigger` based on file sizes
3. Use `maxBytesPerTrigger` to control memory usage
4. Enable `optimizeWrite` and `autoCompact` for Delta tables
5. Monitor stream metrics and adjust trigger intervals
6. Use partitioning in target tables for better query performance
7. Clean up old checkpoint files periodically

---

## Next Steps

1. **Update the file paths** in the examples above with your actual data locations
2. **Start with basic examples** and gradually move to advanced patterns
3. **Test schema evolution** with sample data before production
4. **Set up monitoring** for your Auto Loader streams
5. **Review security** and access permissions

**Happy Streaming with Auto Loader! 🚀**